# MARIDA Retraining — Improved Models with Better Loss Functions
## Trains 3 models with Dice Loss, Class Weighting, Focal Loss, and Strong Augmentation
> All models saved to Google Drive automatically


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['pip', 'install', '-q', 'albumentations', 'scikit-learn', 'tqdm'], check=True)
print("✅ Drive mounted and dependencies installed.")


Mounted at /content/drive
✅ Drive mounted and dependencies installed.


In [2]:
import os, json, torch, numpy as np

MARIDA_PATH = "/content/drive/MyDrive/MARIDA"
SAVE_DIR    = "/content/drive/MyDrive/FYP_models_improved"
os.makedirs(SAVE_DIR, exist_ok=True)

PATCHES_PATH = os.path.join(MARIDA_PATH, "patches")
SPLIT_PATH   = os.path.join(MARIDA_PATH, "splits")
LABELS_FILE  = os.path.join(MARIDA_PATH, "labels_mapping.txt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Models will be saved to: {SAVE_DIR}")

# Training config
EPOCHS = 50
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
PATIENCE = 15  # Early stopping

print(f"Config: {EPOCHS} epochs, batch size {BATCH_SIZE}, LR {LEARNING_RATE}")


Device: cuda
Models will be saved to: /content/drive/MyDrive/FYP_models_improved
Config: 50 epochs, batch size 16, LR 0.0001


In [3]:
import os, json, glob, torch, torch.nn as nn, torch.optim as optim
import numpy as np, rasterio, matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
from sklearn.metrics import f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

SELECTED_CLASSES = ['marine_debris', 'sargassum', 'turbid_water', 'organic', 'cloud']
print("✅ All imports done.")


✅ All imports done.


In [4]:
with open(LABELS_FILE) as f:
    label_dict = json.load(f)
print(f"Loaded {len(label_dict)} label entries.")

# Map 15 classes → 5 classes
target_map = {
    "marine_debris": [0],
    "sargassum":     [3],
    "turbid_water":  [6, 7, 8],
    "organic":       [2],
    "cloud":         [10],
}

filtered_labels = {}
for k, v in label_dict.items():
    vec = [1 if any(v[i]==1 for i in idxs) else 0 for idxs in target_map.values()]
    filtered_labels[k] = vec

counts = np.sum(list(filtered_labels.values()), axis=0)
print(f"\nClass distribution (5-class):")
for cls, c in zip(SELECTED_CLASSES, counts):
    print(f"  {cls:<16}: {int(c):4d} positive patches")

# Compute class weights (inverse frequency)
class_weights = 1.0 / (counts + 1e-6)
class_weights = class_weights / class_weights.sum() * len(class_weights)
print(f"\nClass weights for loss:")
for cls, w in zip(SELECTED_CLASSES, class_weights):
    print(f"  {cls:<16}: {w:.3f}")


Loaded 1381 label entries.

Class distribution (5-class):
  marine_debris   :  373 positive patches
  sargassum       :   71 positive patches
  turbid_water    :  956 positive patches
  organic         :  106 positive patches
  cloud           :   64 positive patches

Class weights for loss:
  marine_debris   : 0.313
  sargassum       : 1.643
  turbid_water    : 0.122
  organic         : 1.100
  cloud           : 1.822


In [5]:
class MARIDAPatchDataset(Dataset):
    def __init__(self, txt_path, patch_dir, label_dict, augment=False):
        with open(txt_path) as f:
            self.image_list = [x.strip() for x in f.readlines()]
        self.patch_dir = patch_dir
        self.labels = label_dict
        self.augment = augment

        # Augmentation pipeline
        self.transform = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Rotate(limit=45, p=0.5),
            A.GaussNoise(p=0.3),
            A.RandomBrightnessContrast(p=0.3),
            A.ToFloat(max_value=1.0),
            ToTensorV2(),
        ], additional_targets={'image': 'image'}) if augment else None

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        while True:
            name = self.image_list[idx]
            scene_id = name.rsplit('_', 1)[0]
            subfolder = f"S2_{scene_id}"
            img_path = os.path.join(self.patch_dir, subfolder, f"S2_{name}.tif")
            label_key = f"S2_{name}.tif"

            if not os.path.exists(img_path):
                idx = np.random.randint(0, len(self.image_list))
                continue
            try:
                with rasterio.open(img_path) as src:
                    img = src.read([1, 2, 3, 4]).astype(np.float32)
            except:
                idx = np.random.randint(0, len(self.image_list))
                continue
            break

        if img.max() > 1:
            img /= 10000.0
        img = np.nan_to_num(img)

        # Spectral indices
        blue, green, red, nir = img[0], img[1], img[2], img[3]
        ndvi = (nir - red) / (nir + red + 1e-6)
        ndwi = (green - nir) / (green + nir + 1e-6)
        fdi = nir - (red + (nir - red) * (833 - 665) / (1610 - 665))

        img = np.stack([blue, green, red, nir, ndvi, ndwi, fdi])  # (7, H, W)
        img = np.transpose(img, (1, 2, 0))  # (H, W, 7)

        if self.augment and self.transform:
            img = self.transform(image=img)['image']
        else:
            img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1)

        label = torch.tensor(self.labels[label_key], dtype=torch.float32)
        return img, label

print("✅ MARIDAPatchDataset defined with strong augmentation.")


✅ MARIDAPatchDataset defined with strong augmentation.


In [6]:
class DiceLoss(nn.Module):
    """Dice/F1 loss for each class."""
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        preds = torch.sigmoid(preds)
        intersection = (preds * targets).sum()
        cardinality = (preds + targets).sum()
        dice_loss = 1 - (2 * intersection + self.smooth) / (cardinality + self.smooth)
        return dice_loss

class FocalLoss(nn.Module):
    """Focal loss for hard examples."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, preds, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(preds, targets, reduction='none')
        p = torch.sigmoid(preds)
        p_t = torch.where(targets == 1, p, 1 - p)
        loss = self.alpha * (1 - p_t) ** self.gamma * bce
        return loss.mean()

class CombinedLoss(nn.Module):
    """Weighted combination: BCE + Dice + Focal."""
    def __init__(self, pos_weight=None, alpha_dice=0.5, alpha_focal=0.3):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.dice = DiceLoss()
        self.focal = FocalLoss()
        self.alpha_dice = alpha_dice
        self.alpha_focal = alpha_focal

    def forward(self, preds, targets):
        bce_loss = self.bce(preds, targets)
        dice_loss = self.dice(preds, targets)
        focal_loss = self.focal(preds, targets)
        return bce_loss + self.alpha_dice * dice_loss + self.alpha_focal * focal_loss

print("✅ Loss functions: BCEWithLogitsLoss + DiceLoss + FocalLoss")


✅ Loss functions: BCEWithLogitsLoss + DiceLoss + FocalLoss


In [7]:
train_dataset = MARIDAPatchDataset(
    os.path.join(SPLIT_PATH, 'train_X.txt'),
    PATCHES_PATH, filtered_labels, augment=True
)
val_dataset = MARIDAPatchDataset(
    os.path.join(SPLIT_PATH, 'val_X.txt'),
    PATCHES_PATH, filtered_labels, augment=False
)
test_dataset = MARIDAPatchDataset(
    os.path.join(SPLIT_PATH, 'test_X.txt'),
    PATCHES_PATH, filtered_labels, augment=False
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} patches | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"Batches: Train {len(train_loader)} | Val {len(val_loader)} | Test {len(test_loader)}")


Train: 339 patches | Val: 73 | Test: 73
Batches: Train 22 | Val 5 | Test 5


In [8]:
class DeepLabV3Classifier(nn.Module):
    def __init__(self, num_classes=5, in_channels=7):
        super().__init__()
        self.deeplab = models.segmentation.deeplabv3_resnet50(weights=None)
        orig = self.deeplab.backbone.conv1
        self.deeplab.backbone.conv1 = nn.Conv2d(
            in_channels, orig.out_channels,
            kernel_size=orig.kernel_size, stride=orig.stride,
            padding=orig.padding, bias=False)
        self.deeplab.classifier = models.segmentation.deeplabv3.DeepLabHead(2048, 256)
        self.classifier = nn.Sequential(
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(256, num_classes))

    def forward(self, x):
        return self.classifier(self.deeplab(x)["out"])

def build_resnet50(num_classes=5, in_channels=7):
    m = models.resnet50(weights=None)
    m.conv1 = nn.Conv2d(in_channels, 64, 7, 2, 3, bias=False)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

def build_efficientnet(num_classes=5, in_channels=7):
    m = models.efficientnet_b0(weights=None)
    fc = m.features[0][0]
    m.features[0][0] = nn.Conv2d(
        in_channels, fc.out_channels,
        kernel_size=fc.kernel_size, stride=fc.stride, padding=fc.padding, bias=False)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    return m

print("✅ Model architectures ready.")


✅ Model architectures ready.


In [13]:
def train_model(model, model_name, train_loader, val_loader, pos_weight, epochs=EPOCHS, lr=LEARNING_RATE):
    """Train one model with early stopping and checkpointing."""

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5)

    # Combined loss with class weighting
    criterion = CombinedLoss(
        pos_weight=pos_weight.to(device),
        alpha_dice=0.5, alpha_focal=0.3
    )

    best_f1 = 0
    patience_counter = 0
    history = {'train_loss': [], 'val_f1': [], 'val_auc': []}

    print(f"\n{'='*70}")
    print(f"  Training {model_name}")
    print(f"{'='*70}")

    for epoch in range(epochs):
        # ─ Train ─
        model.train()
        train_loss = 0
        for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} Train", leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        history['train_loss'].append(train_loss)

        # ─ Validation ─
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for imgs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} Val", leave=False):
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = torch.sigmoid(model(imgs))
                val_preds.append(outputs.cpu().numpy())
                val_true.append(labels.cpu().numpy())

        val_preds = np.vstack(val_preds)
        val_true = np.vstack(val_true)

        # Compute F1 and AUC
        val_pred_binary = (val_preds >= 0.5).astype(int)
        val_f1 = np.mean([f1_score(val_true[:, i], val_pred_binary[:, i], zero_division=0)
                          for i in range(val_true.shape[1])])
        val_auc = np.mean([roc_auc_score(val_true[:, i], val_preds[:, i])
                           if len(np.unique(val_true[:, i])) > 1 else 0.5
                           for i in range(val_true.shape[1])])

        history['val_f1'].append(val_f1)
        history['val_auc'].append(val_auc)

        print(f"  Epoch {epoch+1:2d}  Loss: {train_loss:.4f}  Val F1: {val_f1:.4f}  Val AUC: {val_auc:.4f}", end="")

        # ─ Save checkpoint ─
        if val_f1 > best_f1:
            best_f1 = val_f1
            patience_counter = 0
            ckpt_path = os.path.join(SAVE_DIR, f"{model_name}_best_f1_{val_f1:.3f}.pth")
            torch.save(model.state_dict(), ckpt_path)
            print(f"  ✅ SAVED")
        else:
            patience_counter += 1
            print()

        scheduler.step(val_f1)

        # Early stopping
        if patience_counter >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch+1}")
            break

    # Save final model
    final_path = os.path.join(SAVE_DIR, f"{model_name}_final.pth")
    torch.save(model.state_dict(), final_path)
    print(f"\n  Final model saved: {final_path}")

    return model, history

In [14]:
print("\n🚀 TRAINING MODEL 1 — DeepLabV3")
model1 = DeepLabV3Classifier(num_classes=5, in_channels=7).to(device)
pos_weight_tensor = torch.tensor(class_weights, dtype=torch.float32)
model1, hist1 = train_model(model1, "model1_deeplabv3", train_loader, val_loader, pos_weight_tensor, EPOCHS, LEARNING_RATE)



🚀 TRAINING MODEL 1 — DeepLabV3

  Training model1_deeplabv3


  Epoch  1  Loss: 0.8776  Val F1: 0.1693  Val AUC: 0.4358  ✅ SAVED


  Epoch  2  Loss: 0.7654  Val F1: 0.2553  Val AUC: 0.6333  ✅ SAVED


  Epoch  3  Loss: 0.6964  Val F1: 0.3482  Val AUC: 0.6738  ✅ SAVED


  Epoch  4  Loss: 0.6595  Val F1: 0.2778  Val AUC: 0.7203


...

  Epoch 29  Loss: 0.4903  Val F1: 0.3417  Val AUC: 0.7540


  Epoch 30  Loss: 0.4758  Val F1: 0.3548  Val AUC: 0.7607


  Epoch 31  Loss: 0.4801  Val F1: 0.3548  Val AUC: 0.7752


  Epoch 32  Loss: 0.4952  Val F1: 0.3575  Val AUC: 0.7686


  Epoch 33  Loss: 0.4639  Val F1: 0.4295  Val AUC: 0.7863

  Early stopping at epoch 33

  Final model saved: /content/drive/MyDrive/FYP_models_improved/model1_deeplabv3_final.pth


In [15]:
print("\n🚀 TRAINING MODEL 2 — ResNet50")
model2 = build_resnet50(num_classes=5, in_channels=7).to(device)
model2, hist2 = train_model(model2, "model2_resnet50", train_loader, val_loader, pos_weight_tensor, EPOCHS, LEARNING_RATE)



🚀 TRAINING MODEL 2 — ResNet50

  Training model2_resnet50


  Epoch  1  Loss: 0.7087  Val F1: 0.1587  Val AUC: 0.4065  ✅ SAVED


  Epoch  2  Loss: 0.6203  Val F1: 0.1624  Val AUC: 0.6034  ✅ SAVED


  Epoch  3  Loss: 0.6271  Val F1: 0.1404  Val AUC: 0.6879


  Epoch  4  Loss: 0.6340  Val F1: 0.1569  Val AUC: 0.7177


...

  Epoch 46  Loss: 0.5482  Val F1: 0.2131  Val AUC: 0.7669


  Epoch 47  Loss: 0.5238  Val F1: 0.1510  Val AUC: 0.7977


  Epoch 48  Loss: 0.5274  Val F1: 0.1510  Val AUC: 0.7834


  Epoch 49  Loss: 0.5334  Val F1: 0.1972  Val AUC: 0.7812


  Epoch 50  Loss: 0.5339  Val F1: 0.1970  Val AUC: 0.7747

  Final model saved: /content/drive/MyDrive/FYP_models_improved/model2_resnet50_final.pth


In [16]:
print("\n🚀 TRAINING MODEL 3 — EfficientNet-B0")
model3 = build_efficientnet(num_classes=5, in_channels=7).to(device)
model3, hist3 = train_model(model3, "model3_efficientnet", train_loader, val_loader, pos_weight_tensor, EPOCHS, LEARNING_RATE)



🚀 TRAINING MODEL 3 — EfficientNet-B0

  Training model3_efficientnet


  Epoch  1  Loss: 0.9025  Val F1: 0.1587  Val AUC: 0.5002  ✅ SAVED


  Epoch  2  Loss: 0.7923  Val F1: 0.1587  Val AUC: 0.5840


  Epoch  3  Loss: 0.7058  Val F1: 0.1587  Val AUC: 0.6178


  Epoch  4  Loss: 0.6580  Val F1: 0.1587  Val AUC: 0.5794


...

  Epoch 36  Loss: 0.5603  Val F1: 0.1630  Val AUC: 0.7624


  Epoch 37  Loss: 0.5805  Val F1: 0.1584  Val AUC: 0.7270


  Epoch 38  Loss: 0.5461  Val F1: 0.1658  Val AUC: 0.7566


  Epoch 39  Loss: 0.5592  Val F1: 0.1529  Val AUC: 0.7349


  Epoch 40  Loss: 0.5637  Val F1: 0.1538  Val AUC: 0.7428

  Early stopping at epoch 40

  Final model saved: /content/drive/MyDrive/FYP_models_improved/model3_efficientnet_final.pth


In [17]:
print("\n✅ All models trained! Checking saved files...")
saved_files = os.listdir(SAVE_DIR)
print(f"\nFiles in {SAVE_DIR}:")
for f in sorted(saved_files):
    fsize = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    print(f"  {f:<50s}  ({fsize:6.1f} MB)")

print(f"\nTotal: {len(saved_files)} files saved to Google Drive")



✅ All models trained! Checking saved files...

Files in /content/drive/MyDrive/FYP_models_improved:
  model1_deeplabv3_best_f1_0.169.pth                  ( 159.2 MB)
  model1_deeplabv3_best_f1_0.255.pth                  ( 159.2 MB)
  model1_deeplabv3_best_f1_0.348.pth                  ( 159.2 MB)
  model1_deeplabv3_best_f1_0.349.pth                  ( 159.2 MB)
  model1_deeplabv3_best_f1_0.358.pth                  ( 159.2 MB)
  model1_deeplabv3_best_f1_0.387.pth                  ( 159.2 MB)
  model1_deeplabv3_best_f1_0.453.pth                  ( 159.2 MB)
  model1_deeplabv3_final.pth                          ( 159.2 MB)
  model2_resnet50_best_f1_0.159.pth                   (  94.4 MB)
  model2_resnet50_best_f1_0.162.pth                   (  94.4 MB)
  model2_resnet50_best_f1_0.198.pth                   (  94.4 MB)
  model2_resnet50_best_f1_0.231.pth                   (  94.4 MB)
  model2_resnet50_best_f1_0.250.pth                   (  94.4 MB)
  model2_resnet50_best_f1_0.264.pth      

In [18]:
print("\n🔍 EVALUATION ON TEST SET")
print("\nLoading best model checkpoints...")

def load_best_checkpoint(model, model_name):
    best_ckpts = sorted(glob.glob(os.path.join(SAVE_DIR, f"{model_name}_best_f1_*.pth")))
    if best_ckpts:
        ckpt = best_ckpts[-1]  # Latest (highest F1)
        model.load_state_dict(torch.load(ckpt, map_location=device))
        print(f"  Loaded {os.path.basename(ckpt)}")
    return model

model1 = load_best_checkpoint(model1, "model1_deeplabv3")
model2 = load_best_checkpoint(model2, "model2_resnet50")
model3 = load_best_checkpoint(model3, "model3_efficientnet")

# Evaluate on test set
def evaluate_model(model, test_loader, model_name):
    model.eval()
    y_pred_proba, y_true = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(test_loader, desc=f"Evaluating {model_name}", leave=False):
            imgs = imgs.to(device)
            outputs = torch.sigmoid(model(imgs))
            y_pred_proba.append(outputs.cpu().numpy())
            y_true.append(labels.numpy())

    y_pred_proba = np.vstack(y_pred_proba)
    y_true = np.vstack(y_true)
    y_pred = (y_pred_proba >= 0.5).astype(int)

    f1 = np.mean([f1_score(y_true[:, i], y_pred[:, i], zero_division=0)
                  for i in range(y_true.shape[1])])
    auc = np.mean([roc_auc_score(y_true[:, i], y_pred_proba[:, i])
                   if len(np.unique(y_true[:, i])) > 1 else 0.5
                   for i in range(y_true.shape[1])])
    acc = np.mean((y_pred == y_true).astype(float))

    return {'f1': f1, 'auc': auc, 'acc': acc}

print()
res1 = evaluate_model(model1, test_loader, "Model 1")
res2 = evaluate_model(model2, test_loader, "Model 2")
res3 = evaluate_model(model3, test_loader, "Model 3")

print("\n" + "="*70)
print(f"  {'Model':<25} {'Accuracy':>10} {'F1 (Macro)':>12} {'AUC (Macro)':>12}")
print("="*70)
print(f"  {'DeepLabV3':<25} {res1['acc']*100:9.2f}%  {res1['f1']:11.4f}  {res1['auc']:11.4f}")
print(f"  {'ResNet50':<25} {res2['acc']*100:9.2f}%  {res2['f1']:11.4f}  {res2['auc']:11.4f}")
print(f"  {'EfficientNet':<25} {res3['acc']*100:9.2f}%  {res3['f1']:11.4f}  {res3['auc']:11.4f}")
print("="*70)

print(f"\n✅ Models are ready in Google Drive at: {SAVE_DIR}")
print("   Use these models in your evaluation notebook!")



🔍 EVALUATION ON TEST SET

Loading best model checkpoints...
  Loaded model1_deeplabv3_best_f1_0.453.pth
  Loaded model2_resnet50_best_f1_0.277.pth
  Loaded model3_efficientnet_best_f1_0.172.pth




  Model                       Accuracy   F1 (Macro)  AUC (Macro)
  DeepLabV3                     86.58%       0.3684       0.7347
  ResNet50                      85.21%       0.2745       0.7602
  EfficientNet                  84.66%       0.1635       0.6908

✅ Models are ready in Google Drive at: /content/drive/MyDrive/FYP_models_improved
   Use these models in your evaluation notebook!


## Training Complete! ✅

Your improved models have been saved to Google Drive:
- `model1_deeplabv3_final.pth`
- `model2_resnet50_final.pth`
- `model3_efficientnet_final.pth`

### Improvements Applied:
✅ **Dice Loss** — Per-class F1 optimization  
✅ **Focal Loss** — Focus on hard examples  
✅ **Class Weighting** — Rare classes weighted higher  
✅ **Strong Augmentation** — Flips, rotations, Gaussian noise  
✅ **Lower Learning Rate** — 1e-4 with schedule  
✅ **Early Stopping** — Avoid overfitting  
✅ **Automatic Checkpointing** — Save best models to Drive  

### Next Steps:
1. Download the paths from Drive or use them directly in Colab
2. Use `Model_Copy_of_Working_FYP_1.ipynb` for evaluation with TTA + ensemble
3. Generate comprehensive metrics and visualizations

Good luck with your FYP! 🎓
